# XML, JSON y APIs — Version Interactiva
## ISS en el Mapa con Folium

Esta version expande el notebook principal con visualizaciones interactivas de la posicion
de la Estacion Espacial Internacional usando mapas en tiempo real.

**Temas adicionales:**
- Instalar librerias directamente desde el notebook
- Visualizar datos geograficos con **Folium**
- Tomar multiples lecturas de una API y renderizar una trayectoria
- Calcular velocidad aproximada a partir de coordenadas (formula de Haversine)

---
## Instalar librerias desde el notebook

Normalmente instalas librerias desde la terminal con `pip install nombre`. Pero tambien
puedes hacerlo **directamente dentro de una celda de Jupyter** usando el prefijo `!` o `%pip`.

### `!pip install` vs `%pip install`

| Comando | Como funciona | Cuando usarlo |
|---|---|---|
| `!pip install lib` | Ejecuta `pip` en el shell del sistema | Instalaciones rapidas |
| `%pip install lib` | Instala en el mismo kernel del notebook | **Recomendado en Jupyter** |

> **Por que importa la diferencia?**
> Si tienes varios entornos (conda, venv), `!pip` puede instalar en el Python del sistema
> en lugar del que usa el notebook. `%pip` siempre instala donde corre el kernel activo.

### Otras magics utiles en Jupyter

| Magic | Descripcion |
|---|---|
| `%pip install lib` | Instalar libreria |
| `%pip list` | Ver todas las librerias instaladas |
| `%pip show lib` | Ver detalles de una libreria |
| `!comando` | Ejecutar cualquier comando de terminal |
| `%time codigo` | Medir el tiempo de ejecucion de una linea |
| `%%time` | Medir el tiempo de ejecucion de toda la celda |

In [1]:
# Verificar si folium esta instalado; si no, instalarlo automaticamente
try:
    import folium
    print(f"folium ya instalado -- version {folium.__version__}")
except ImportError:
    print("Instalando folium...")
    %pip install folium
    import folium
    print(f"folium instalado -- version {folium.__version__}")

folium ya instalado -- version 0.20.0


In [2]:
import requests
import time
import math
import json
from datetime import datetime, timezone
from IPython.display import display

print("Librerias importadas correctamente")

Librerias importadas correctamente


---
## Ejemplo 1 -- ISS en el mapa

### Parte A -- Posicion actual

Obtenemos la posicion actual de la ISS y la mostramos en un mapa interactivo con Folium.

**Folium** genera mapas HTML interactivos (basados en Leaflet.js) que se renderizan
directamente en el notebook. Puedes hacer zoom, arrastrar y hacer clic en los marcadores.

El endpoint `http://api.open-notify.org/iss-now.json` no requiere API key y devuelve
la posicion actualizada cada vez que lo consultas.

In [3]:
# Obtener posicion actual de la ISS
url_iss = "http://api.open-notify.org/iss-now.json"
r = requests.get(url_iss, timeout=15)

if not r.ok:
    raise SystemExit(f"Error {r.status_code}")

datos = r.json()
lat  = float(datos["iss_position"]["latitude"])
lon  = float(datos["iss_position"]["longitude"])
ts   = datos["timestamp"]
hora = datetime.fromtimestamp(ts, tz=timezone.utc).strftime("%Y-%m-%d %H:%M:%S UTC")

print(f"Posicion actual de la ISS")
print(f"  Latitud  : {lat:+.4f}")
print(f"  Longitud : {lon:+.4f}")
print(f"  Hora UTC : {hora}")

Posicion actual de la ISS
  Latitud  : -43.1378
  Longitud : -0.0777
  Hora UTC : 2026-04-17 17:25:53 UTC


In [4]:
# Crear mapa centrado en la posicion actual de la ISS
mapa = folium.Map(location=[lat, lon], zoom_start=3, tiles="CartoDB positron")

# Texto del popup
popup_texto = (
    "<b>ISS -- Estacion Espacial Internacional</b><br>"
    f"<b>Hora:</b> {hora}<br>"
    f"<b>Lat:</b> {lat:+.4f}  <b>Lon:</b> {lon:+.4f}<br>"
    "<b>Altitud orbital:</b> ~408 km<br>"
    "<b>Velocidad orbital:</b> ~27,600 km/h"
)

# Marcador con icono de estrella
folium.Marker(
    location=[lat, lon],
    popup=folium.Popup(popup_texto, max_width=280),
    tooltip="ISS -- clic para mas info",
    icon=folium.Icon(color="red", icon="star", prefix="fa"),
).add_to(mapa)

# Circulo que representa la zona visible desde la ISS (~2200 km de radio)
folium.Circle(
    location=[lat, lon],
    radius=2_200_000,          # en metros
    color="steelblue",
    fill=True,
    fill_opacity=0.08,
    tooltip="Zona visible desde la ISS",
).add_to(mapa)

display(mapa)

---
### Parte B -- Trayectoria y velocidad

La ISS orbita la Tierra a **~7.66 km/s** -- completa una orbita en ~92 minutos.

Vamos a tomar varias lecturas separadas por unos segundos, calcular la distancia
entre puntos consecutivos y estimar la velocidad usando la **formula de Haversine**.

#### Formula de Haversine

Calcula la distancia entre dos puntos sobre la superficie de una esfera a partir
de sus coordenadas de latitud y longitud:

$$d = 2R \cdot \arctan2\left(\sqrt{a},\, \sqrt{1-a}\right)$$

donde:

$$a = \sin^2\!\left(\frac{\Delta\phi}{2}\right) + \cos\phi_1 \cdot \cos\phi_2 \cdot \sin^2\!\left(\frac{\Delta\lambda}{2}\right)$$

y $R = 6{,}371$ km es el radio terrestre.

In [5]:
def haversine(lat1, lon1, lat2, lon2):
    """
    Calcula la distancia en km entre dos puntos sobre la superficie terrestre.
    Los angulos se dan en grados decimales.
    """
    R = 6_371  # radio de la Tierra en km

    phi1    = math.radians(lat1)
    phi2    = math.radians(lat2)
    dphi    = math.radians(lat2 - lat1)
    dlambda = math.radians(lon2 - lon1)

    a = (math.sin(dphi / 2) ** 2
         + math.cos(phi1) * math.cos(phi2) * math.sin(dlambda / 2) ** 2)

    return R * 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))


# Prueba: Ciudad de Guatemala -> Ciudad de Mexico (~930 km)
d = haversine(14.64, -90.51, 19.43, -99.13)
print(f"Prueba haversine: Guatemala -> CDMX = {d:.0f} km  (esperado ~930 km)")

Prueba haversine: Guatemala -> CDMX = 1060 km  (esperado ~930 km)


In [7]:
#############################################################
############# funcion implementada en clase #################
#############################################################

# Configuracion del rastreo
N_LECTURAS   = 5    # lecturas exitosas que queremos obtener
PAUSA        = 10   # segundos entre lecturas
MAX_INTENTOS = 15   # intentos maximos para evitar bucle infinito si la API falla mucho
timeout_request = 5

lecturas  = []
intentos_exitosos  = 0

url_iss = "http://api.open-notify.org/iss-now.json"

for i in range(16):
    print(f"intento {i}")

    try:
        r = requests.get(url_iss, timeout=timeout_request)
        
        if not r.ok:
            raise SystemExit(f"Error {r.status_code}")
        else:
            intentos_exitosos += 1
            ## Extraer datos
            '''
            {'message': 'success',
              'iss_position': {'longitude': '104.7152', 'latitude': '-47.4126'},
              'timestamp': 1776269139}
            '''
            raw_data = r.json()
            lectura = {
                "lat" : float(raw_data['iss_position']['latitude']),
                "long" : float(raw_data['iss_position']['longitude']),
                "timestamp": raw_data['timestamp']
            }
            ## Almacenar en la lista
            lecturas.append(lectura)
    
        if intentos_exitosos == N_LECTURAS:
            break
            
        time.sleep(2)
    except requests.exceptions.Timeout:
        print(f"   Intento: {i}  Error de Timeout")
        timeout_request += 1

if len(lecturas) == N_LECTURAS:
    print(f"{N_LECTURAS} Exitosas")

intento 0
   Intento: 0  Error de Timeout
intento 1
intento 2
intento 3
intento 4
   Intento: 4  Error de Timeout
intento 5
intento 6
5 Exitosas


In [8]:
lecturas

[{'lat': -43.3238, 'long': 0.3149, 'timestamp': 1776446758},
 {'lat': -43.4078, 'long': 0.4941, 'timestamp': 1776446761},
 {'lat': -43.4915, 'long': 0.674, 'timestamp': 1776446763},
 {'lat': -43.8067, 'long': 1.3618, 'timestamp': 1776446773},
 {'lat': -43.8889, 'long': 1.5444, 'timestamp': 1776446775}]

In [9]:
# Configuracion del rastreo
N_LECTURAS   = 5    # lecturas exitosas que queremos obtener
PAUSA        = 10   # segundos entre lecturas
MAX_INTENTOS = 15   # intentos maximos para evitar bucle infinito si la API falla mucho

lecturas  = []
intentos  = 0

print(f"Buscando {N_LECTURAS} lecturas exitosas (max {MAX_INTENTOS} intentos, {PAUSA}s entre c/u)...")
print()
print(f"{'#':>3}  {'Hora UTC':<21} {'Lat':>9} {'Lon':>10}  {'Dist km':>9}  {'Vel km/h':>10}")
print("-" * 70)

while len(lecturas) < N_LECTURAS and intentos < MAX_INTENTOS:
    intentos += 1
    try:
        r = requests.get("http://api.open-notify.org/iss-now.json", timeout=8)
        r.raise_for_status()
        d = r.json()

        lat_i  = float(d["iss_position"]["latitude"])
        lon_i  = float(d["iss_position"]["longitude"])
        ts_i   = d["timestamp"]
        hora_i = datetime.fromtimestamp(ts_i, tz=timezone.utc).strftime("%H:%M:%S UTC")

        lectura = {"lat": lat_i, "lon": lon_i, "ts": ts_i, "hora": hora_i,
                   "dist_km": None, "vel_kmh": None}

        if lecturas:
            prev     = lecturas[-1]
            dist_km  = haversine(prev["lat"], prev["lon"], lat_i, lon_i)
            dt_horas = (ts_i - prev["ts"]) / 3600
            vel_kmh  = dist_km / dt_horas if dt_horas > 0 else 0
            lectura["dist_km"] = dist_km
            lectura["vel_kmh"] = vel_kmh
            print(f"{len(lecturas)+1:>3}  {hora_i:<21} {lat_i:>+9.4f} {lon_i:>+10.4f}  "
                  f"{dist_km:>9.2f}  {vel_kmh:>10,.0f}")
        else:
            print(f"{len(lecturas)+1:>3}  {hora_i:<21} {lat_i:>+9.4f} {lon_i:>+10.4f}  "
                  f"{'primera':>9}  {'lectura':>10}")

        lecturas.append(lectura)

    except requests.exceptions.Timeout:
        print(f"  [intento {intentos}] Timeout -- reintentando en {PAUSA}s...")
    except requests.exceptions.ConnectionError:
        print(f"  [intento {intentos}] Sin conexion -- reintentando en {PAUSA}s...")
    except requests.exceptions.RequestException as e:
        print(f"  [intento {intentos}] Error: {e} -- reintentando en {PAUSA}s...")

    if len(lecturas) < N_LECTURAS and intentos < MAX_INTENTOS:
        time.sleep(PAUSA)

if len(lecturas) < N_LECTURAS:
    print(f"\nAtencion: solo se obtuvieron {len(lecturas)} de {N_LECTURAS} lecturas tras {intentos} intentos.")
else:
    print(f"\nListo -- {len(lecturas)} lecturas exitosas en {intentos} intento(s)")

Buscando 5 lecturas exitosas (max 15 intentos, 10s entre c/u)...

  #  Hora UTC                    Lat        Lon    Dist km    Vel km/h
----------------------------------------------------------------------
  1  17:26:16 UTC           -43.9053    +1.5809    primera     lectura
  2  17:26:26 UTC           -44.2465    +2.3527      72.39      26,061
  [intento 3] Timeout -- reintentando en 10s...
  3  17:26:58 UTC           -45.2343    +4.7253     217.20      24,435
  4  17:27:12 UTC           -45.6405    +5.7685      93.09      23,936
  5  17:27:22 UTC           -45.9491    +6.5909      72.41      26,066

Listo -- 5 lecturas exitosas en 6 intento(s)


In [10]:
# Calcular centro del mapa como promedio de posiciones
centro_lat = sum(l["lat"] for l in lecturas) / len(lecturas)
centro_lon = sum(l["lon"] for l in lecturas) / len(lecturas)

mapa_traj = folium.Map(
    location=[centro_lat, centro_lon],
    zoom_start=4,
    tiles="CartoDB dark_matter"
)

# Linea de trayectoria
coords = [[l["lat"], l["lon"]] for l in lecturas]
folium.PolyLine(
    locations=coords,
    color="#00d4ff",
    weight=2.5,
    opacity=0.85,
    tooltip="Trayectoria ISS",
).add_to(mapa_traj)

# Marcador por cada lectura
for i, l in enumerate(lecturas):
    vel_str  = f"{l['vel_kmh']:,.0f} km/h" if l["vel_kmh"] else "primera lectura"
    dist_str = f"{l['dist_km']:.2f} km"   if l["dist_km"] else "N/A"

    # Verde = primera, Rojo = ultima, Azul = intermedias
    if i == 0:
        color, icono = "green", "play"
    elif i == len(lecturas) - 1:
        color, icono = "red", "stop"
    else:
        color, icono = "cadetblue", "circle"

    popup_html = (
        f"<b>Lectura {i + 1} de {len(lecturas)}</b><br>"
        f"<b>Hora:</b> {l['hora']}<br>"
        f"<b>Lat:</b> {l['lat']:+.4f}  <b>Lon:</b> {l['lon']:+.4f}<br>"
        f"<b>Dist. desde anterior:</b> {dist_str}<br>"
        f"<b>Velocidad aprox.:</b> {vel_str}"
    )

    folium.Marker(
        location=[l["lat"], l["lon"]],
        popup=folium.Popup(popup_html, max_width=260),
        tooltip=f"Lectura {i+1} -- {l['hora']}",
        icon=folium.Icon(color=color, icon=icono, prefix="fa"),
    ).add_to(mapa_traj)

# Zona visible desde la ultima posicion
ultima = lecturas[-1]
folium.Circle(
    location=[ultima["lat"], ultima["lon"]],
    radius=2_200_000,
    color="#00d4ff",
    fill=True,
    fill_opacity=0.06,
    tooltip="Zona visible desde la ISS (ultima posicion)",
).add_to(mapa_traj)

display(mapa_traj)

In [11]:
# Resumen estadistico del rastreo
velocidades = [l["vel_kmh"] for l in lecturas if l["vel_kmh"] is not None]
distancias  = [l["dist_km"] for l in lecturas if l["dist_km"] is not None]

duracion_s       = lecturas[-1]["ts"] - lecturas[0]["ts"]
distancia_total  = sum(distancias)
vel_promedio     = sum(velocidades) / len(velocidades)

print("Resumen del rastreo")
print("-" * 45)
print(f"  Lecturas          : {len(lecturas)}")
print(f"  Duracion total    : {duracion_s} segundos")
print(f"  Distancia total   : {distancia_total:,.1f} km")
print(f"  Velocidad promedio: {vel_promedio:,.0f} km/h")
print(f"  Velocidad maxima  : {max(velocidades):,.0f} km/h")
print(f"  Velocidad minima  : {min(velocidades):,.0f} km/h")
print()
print("  Nota: la velocidad real de la ISS es ~27,600 km/h (~7.66 km/s).")
print("  Las variaciones se deben a que el API actualiza cada pocos segundos")
print("  y a la precision finita de las coordenadas reportadas.")

Resumen del rastreo
---------------------------------------------
  Lecturas          : 5
  Duracion total    : 66 segundos
  Distancia total   : 455.1 km
  Velocidad promedio: 25,125 km/h
  Velocidad maxima  : 26,066 km/h
  Velocidad minima  : 23,936 km/h

  Nota: la velocidad real de la ISS es ~27,600 km/h (~7.66 km/s).
  Las variaciones se deben a que el API actualiza cada pocos segundos
  y a la precision finita de las coordenadas reportadas.
